In [2]:
import os
import random
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
df = pd.read_csv("data/03_panel_instability.csv")
df = df.sort_values(["COUNTRY", "YEAR"]).copy()

base_feature_cols = [
    "GDP_Growth_lag1",
    "GDP_Growth_rollmean3",
    "Inflation_lag1_log",
    "Exports_lag1",
    "Imports_lag1",
    "Fiscal_Balance_lag1",
    "Current_Account_lag1",
    "Debt_diff_lag1",
    "Expenditure_diff_lag1",
    "Revenue_diff_lag1",
    "Savings_diff_lag1",
    "Investment_diff_lag1",
    "Instability_Index_lag1",
]

volatility_cols = [
    col for col in [
        "GDP_Growth_rollstd3",
        "Inflation_rollstd3",
        "Exports_rollstd3",
        "Imports_rollstd3",
        "Fiscal_Balance_rollstd3",
        "Current_Account_rollstd3",
        "Debt_rollstd3",
        "Revenue_rollstd3",
        "Expenditure_rollstd3",
        "Savings_rollstd3",
        "Investment_rollstd3",
    ]
    if col in df.columns
]

# Same useful country-history features used in improved ML layer
df["Country_GDP_Mean_lag1"] = (
    df.groupby("COUNTRY")["GDP_Growth"]
    .transform(lambda x: x.shift(1).expanding(min_periods=3).mean())
)

df["Country_GDP_Std_lag1"] = (
    df.groupby("COUNTRY")["GDP_Growth"]
    .transform(lambda x: x.shift(1).expanding(min_periods=3).std())
)

df["GDP_vs_Country_Mean_lag1"] = (
    df["GDP_Growth_lag1"] - df["Country_GDP_Mean_lag1"]
)

df["Year_Trend"] = df["YEAR"] - df["YEAR"].min()

country_history_cols = [
    "Country_GDP_Mean_lag1",
    "Country_GDP_Std_lag1",
    "GDP_vs_Country_Mean_lag1",
    "Year_Trend",
]

feature_cols = (
    base_feature_cols
    + volatility_cols
    + country_history_cols
)

df_model = df.dropna(
    subset=feature_cols + ["GDP_Growth"]
).copy()

print(f"Loaded: {df_model.shape}")
print(f"Countries: {df_model['COUNTRY'].nunique()}")
print(f"Years: {df_model['YEAR'].min()}-{df_model['YEAR'].max()}")
print(f"Features: {len(feature_cols)}")

Loaded: (4632, 120)
Countries: 175
Years: 2000-2026
Features: 28


In [4]:
SEQ_LEN = 5

def build_sequences(data, feature_cols, target_col, seq_len=5):
    X_seq = []
    y_seq = []
    meta = []

    for country, grp in data.groupby("COUNTRY"):
        grp = grp.sort_values("YEAR").reset_index(drop=True)

        X_values = grp[feature_cols].to_numpy()
        y_values = grp[target_col].to_numpy()
        years = grp["YEAR"].to_numpy()

        for i in range(seq_len - 1, len(grp)):
            X_seq.append(
                X_values[i - seq_len + 1:i + 1]
            )

            y_seq.append(y_values[i])

            meta.append({
                "COUNTRY": country,
                "YEAR": years[i],
            })

    return (
        np.array(X_seq),
        np.array(y_seq),
        pd.DataFrame(meta),
    )


X_all, y_all, meta_all = build_sequences(
    df_model,
    feature_cols,
    "GDP_Growth",
    seq_len=SEQ_LEN,
)

print("X_all:", X_all.shape)
print("y_all:", y_all.shape)
print(meta_all.head())

X_all: (3932, 5, 28)
y_all: (3932,)
                            COUNTRY  YEAR
0  Afghanistan, Islamic Republic of  2012
1  Afghanistan, Islamic Republic of  2013
2  Afghanistan, Islamic Republic of  2014
3  Afghanistan, Islamic Republic of  2015
4  Afghanistan, Islamic Republic of  2016


In [5]:
train_mask = meta_all["YEAR"] <= 2019
covid_mask = meta_all["YEAR"].isin([2020, 2021])
test_mask = meta_all["YEAR"].isin([2022, 2023])
projection_mask = meta_all["YEAR"].isin([2024, 2025, 2026])

X_train = X_all[train_mask]
y_train = y_all[train_mask]

X_covid = X_all[covid_mask]
y_covid = y_all[covid_mask]

X_test = X_all[test_mask]
y_test = y_all[test_mask]
meta_test = meta_all[test_mask].reset_index(drop=True)

X_projection = X_all[projection_mask]
meta_projection = meta_all[projection_mask].reset_index(drop=True)

print(f"Train: {X_train.shape}")
print(f"COVID stress: {X_covid.shape}")
print(f"Observed test 2022-23: {X_test.shape}")
print(f"Projection 2024-26: {X_projection.shape}")

Train: (2728, 5, 28)
COVID stress: (345, 5, 28)
Observed test 2022-23: (346, 5, 28)
Projection 2024-26: (513, 5, 28)


In [6]:
n_features = X_train.shape[2]

scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, n_features)
scaler.fit(X_train_2d)

def scale_sequences(X, scaler):
    original_shape = X.shape
    X_2d = X.reshape(-1, original_shape[2])
    X_scaled = scaler.transform(X_2d)
    return X_scaled.reshape(original_shape)

X_train_sc = scale_sequences(X_train, scaler)
X_covid_sc = scale_sequences(X_covid, scaler)
X_test_sc = scale_sequences(X_test, scaler)
X_projection_sc = scale_sequences(X_projection, scaler)

In [7]:
model = Sequential([
    LSTM(
        32,
        input_shape=(SEQ_LEN, n_features),
        return_sequences=False,
    ),
    Dropout(0.25),
    Dense(16, activation="relu"),
    Dense(1),
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"],
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         7,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,353 (32.63 KB)

 Trainable params: 8,353 (32.63 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
)

history = model.fit(
    X_train_sc,
    y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)

model.save("models/lstm_gdp_growth.keras")

print("Saved: models/lstm_gdp_growth.keras")

Epoch 1/200
69/69 ━━━━━━━━━━━━━━━━━━━━ 9s 27ms/step - loss: 32.5371 - mae: 4.0781 - val_loss: 23.1867 - val_mae: 3.4838
Epoch 2/200
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 22.6890 - mae: 2.8806 - val_loss: 17.0244 - val_mae: 2.6203
Epoch 3/200
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 20.3694 - mae: 2.5769 - val_loss: 16.2581 - val_mae: 2.5090
Epoch 4/200
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 19.6181 - mae: 2.5329 - val_loss: 15.8118 - val_mae: 2.4415
Epoch 5/200
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 19.0812 - mae: 2.4752 - val_loss: 15.4201 - val_mae: 2.3941
Epoch 6/200
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 18.4401 - mae: 2.4266 - val_loss: 15.0785 - val_mae: 2.3563
Epoch 7/200
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 18.1968 - mae: 2.3940 - val_loss: 14.8275 - val_mae: 2.3290
Epoch 8/200
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 17.6621 - mae: 2.3800 - val_loss: 14.6582 - val_mae: 2.3193
Epoch 9/200
69/69 ━━━━━━━━━━━━━━━━━━━━ 1

In [9]:
def evaluate_regression(name, y_true, y_pred, period):
    return {
        "Model": name,
        "Period": period,
        "N": len(y_true),
        "RMSE": round(
            np.sqrt(mean_squared_error(y_true, y_pred)),
            3,
        ),
        "MAE": round(
            mean_absolute_error(y_true, y_pred),
            3,
        ),
        "R2": round(
            r2_score(y_true, y_pred),
            3,
        ),
        "Mean_Actual": round(float(np.mean(y_true)), 2),
        "Mean_Predicted": round(float(np.mean(y_pred)), 2),
        "Mean_Error": round(float(np.mean(y_true - y_pred)), 2),
    }


pred_test = model.predict(X_test_sc).flatten()

lstm_rows = []

lstm_rows.append(
    evaluate_regression(
        "LSTM",
        y_test,
        pred_test,
        "Observed test (2022-23)",
    )
)

for year in [2022, 2023]:
    mask = meta_test["YEAR"] == year

    lstm_rows.append(
        evaluate_regression(
            "LSTM",
            y_test[mask],
            pred_test[mask],
            f"Year {year}",
        )
    )

lstm_results_df = pd.DataFrame(lstm_rows)

lstm_results_df.to_csv(
    "data/layer3_lstm_results.csv",
    index=False,
)

display(lstm_results_df)
print("Saved: data/layer3_lstm_results.csv")

11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step


,Model,Period,N,RMSE,MAE,R2,Mean_Actual,Mean_Predicted,Mean_Error
0,LSTM,Observed test (2022-23),346,5.117,2.771,0.104,3.81,3.34,0.48
1,LSTM,Year 2022,173,6.229,3.246,0.073,4.51,3.29,1.22
2,LSTM,Year 2023,173,3.683,2.295,0.130,3.12,3.38,-0.27


Saved: data/layer3_lstm_results.csv


In [10]:
pred_covid = model.predict(X_covid_sc).flatten()

lstm_covid_df = pd.DataFrame([
    evaluate_regression(
        "LSTM",
        y_covid,
        pred_covid,
        "COVID stress test (2020-21)",
    )
])

lstm_covid_df.to_csv(
    "data/layer3_lstm_covid_stress.csv",
    index=False,
)

display(lstm_covid_df)
print("Saved: data/layer3_lstm_covid_stress.csv")

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


,Model,Period,N,RMSE,MAE,R2,Mean_Actual,Mean_Predicted,Mean_Error
0,LSTM,COVID stress test (2020-21),345,8.788,6.621,-0.111,0.79,2.37,-1.58


Saved: data/layer3_lstm_covid_stress.csv


In [11]:
pred_projection = model.predict(X_projection_sc).flatten()

lstm_projection_df = meta_projection.copy()
lstm_projection_df["LSTM_Forecast"] = pred_projection

lstm_projection_df.to_csv(
    "data/layer3_lstm_forecasts_2024_2026.csv",
    index=False,
)

display(lstm_projection_df.head())
print("Saved: data/layer3_lstm_forecasts_2024_2026.csv")

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


,COUNTRY,YEAR,LSTM_Forecast
0,"Afghanistan, Islamic Republic of",2024,2.438284
1,"Afghanistan, Islamic Republic of",2025,3.608327
2,Albania,2024,5.029444
3,Albania,2025,5.140644
4,Albania,2026,5.189594


Saved: data/layer3_lstm_forecasts_2024_2026.csv


In [12]:
layer2b_best = pd.read_csv("data/layer2b_best_result.csv")
lstm_results = pd.read_csv("data/layer3_lstm_results.csv")

lstm_main = lstm_results[
    lstm_results["Period"] == "Observed test (2022-23)"
].copy()

comparison = pd.DataFrame([
    {
        "Layer": "Layer 2b ML",
        "Model": layer2b_best.loc[0, "Model"],
        "RMSE": layer2b_best.loc[0, "RMSE"],
        "MAE": layer2b_best.loc[0, "MAE"],
        "R2": layer2b_best.loc[0, "R2"],
    },
    {
        "Layer": "Layer 3 LSTM",
        "Model": "LSTM",
        "RMSE": lstm_main.iloc[0]["RMSE"],
        "MAE": lstm_main.iloc[0]["MAE"],
        "R2": lstm_main.iloc[0]["R2"],
    },
])

comparison = comparison.sort_values("RMSE")

comparison.to_csv(
    "data/layer2b_vs_lstm_comparison.csv",
    index=False,
)

display(comparison)
print("Saved: data/layer2b_vs_lstm_comparison.csv")

,Layer,Model,RMSE,MAE,R2
0,Layer 2b ML,Random Forest,4.709,2.631,0.241
1,Layer 3 LSTM,LSTM,5.117,2.771,0.104


Saved: data/layer2b_vs_lstm_comparison.csv
